# Oracle metadata extraction

Load all canonical Pydantic models. Tables contain columns, constraints, and indexes; views contain columns and dependencies. Edit the connection and schema below for your database.

In [ ]:
import json
from pathlib import Path

# Run the notebook from the project root; see config.example.json.
config = json.loads(Path("config.json").read_text())
if not config.get("OPENMETADATA_JWT_TOKEN", "").strip():
    raise ValueError("Set OPENMETADATA_JWT_TOKEN in config.json.")


In [ ]:
from sqlalchemy import create_engine

from core.extract.OracleExtractor import OracleExtractor

schema_name = "DEVELOPER"
engine = create_engine(config["METADATA_MINER_ORACLE_URL"])
extractor = OracleExtractor(engine)



In [ ]:
from datetime import datetime, timezone

try:
    users = extractor.extract_users(schema_name)
    objects = extractor.extract_objects(schema_name)
    tables = extractor.extract_tables(schema_name)
    columns = extractor.extract_columns(schema_name)
    views = extractor.extract_views(schema_name)
    indexes = extractor.extract_indexes(schema_name)
    triggers = extractor.extract_triggers(schema_name)
    sequences = extractor.extract_sequences(schema_name)
    synonyms = extractor.extract_synonyms(schema_name)
    table_statistics = extractor.extract_table_statistics(schema_name)
    column_statistics = extractor.extract_column_statistics(schema_name)
    histograms = extractor.extract_histograms(schema_name)
    constraints = extractor.extract_constraints(schema_name)
    view_dependencies = extractor.extract_view_dependencies(schema_name)
finally:
    engine.dispose()

extracted_at_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
canonical_models = {
    "users": users,
    "objects": objects,
    "tables": tables,
    "columns": columns,
    "views": views,
    "indexes": indexes,
    "triggers": triggers,
    "sequences": sequences,
    "synonyms": synonyms,
    "table_statistics": table_statistics,
    "column_statistics": column_statistics,
    "histograms": histograms,
    "constraints": constraints,
    "view_dependencies": view_dependencies,
}
extraction_info = extractor.extraction_info

{name: len(models) for name, models in canonical_models.items()}


## OpenMetadata mappings

Create SDK payloads locally using `OracleMapper`. Set `database_fqn` to your OpenMetadata database. Profile timestamps represent extraction time; optimizer statistics can be older.

Lineage requires actual server entity IDs in `entity_ids`; unresolved dependencies remain in `pending_lineage`. Remote synonyms remain in `pending_remote_synonyms`. CHECK conditions, indexes, triggers, sequences, generic objects, and raw histogram endpoints remain available as canonical metadata where the mapper has no direct target. These cells do not send payloads to a server.

In [ ]:
from collections import defaultdict
from uuid import UUID

from core.cdm import ConstraintMetadata
from core.mapping.open_metadata import OracleMapper

# Use the service.database FQN configured in your OpenMetadata server.
database_fqn = "oracle.library"
mapper = OracleMapper(database_fqn)

constraint_groups: dict[tuple[str, str, str], list[ConstraintMetadata]] = defaultdict(list)
for constraint in constraints:
    key = (constraint.schema_name, constraint.table_name, constraint.name)
    constraint_groups[key].append(constraint)

mapped_constraints = []
for rows in constraint_groups.values():
    mapped = mapper.map_constraint(rows)
    if mapped is not None:
        mapped_constraints.append(mapped)

# Populate this registry with actual OpenMetadata entity UUIDs to create lineage.
# Keys are (schema_name, object_name); views are OpenMetadata table entities.
entity_ids: dict[tuple[str, str], UUID] = {}
mapped_lineage = []
pending_lineage = []
for dependency in view_dependencies:
    source_key = (dependency.referenced_schema_name, dependency.referenced_object_name)
    view_key = (dependency.schema_name, dependency.view_name)
    if (
        dependency.referenced_object_type in {"TABLE", "VIEW", "MATERIALIZED VIEW"}
        and source_key in entity_ids
        and view_key in entity_ids
    ):
        mapped_lineage.append(mapper.map_view_dependency(
            dependency, entity_ids[source_key], entity_ids[view_key]
        ))
    else:
        pending_lineage.append(dependency)

mappings = {
    "schemas": [mapper.map_schema(model) for model in users],
    "tables": [mapper.map_table(model) for model in tables],
    "views": [mapper.map_view(model) for model in views],
    "columns": [mapper.map_column(model) for model in columns],
    "constraints": mapped_constraints,
    "table_profiles": [
        mapper.map_table_statistics(model, extracted_at_ms)
        for model in table_statistics if model.object_type == "TABLE"
    ],
    "column_profiles": [
        mapper.map_column_statistics(model, extracted_at_ms)
        for model in column_statistics
    ],
    "aliases": [mapper.map_synonym(model) for model in synonyms if not model.db_link],
    "lineage": mapped_lineage,
}
pending_remote_synonyms = [model for model in synonyms if model.db_link]

{
    "mapped": {name: len(models) for name, models in mappings.items()},
    "pending_lineage": len(pending_lineage),
    "pending_remote_synonyms": len(pending_remote_synonyms),
}


## Publish to OpenMetadata

Running this cell creates or updates schemas, tables/views, aliases, profiles, and lineage in your configured server. Set `OPENMETADATA_JWT_TOKEN` and optionally `OPENMETADATA_URL` in root `config.json` (loaded by the first code cell; see `config.example.json`), and set `database_fqn`. The publishing cell creates the Oracle service and database if missing. Columns and constraints are included in table requests. Dependencies outside the extracted schema remain pending unless their server IDs are supplied in `entity_ids`.

In [ ]:
from collections import defaultdict

from metadata.generated.schema.entity.data.table import ColumnProfile
from metadata.generated.schema.entity.services.connections.metadata.openMetadataConnection import OpenMetadataConnection
from metadata.ingestion.ometa.ometa_api import OpenMetadata

from core.publish import OpenMetadataPublisher
from metadata.generated.schema.api.services.createDatabaseService import CreateDatabaseServiceRequest
from metadata.generated.schema.api.data.createDatabase import CreateDatabaseRequest
from metadata.generated.schema.entity.services.databaseService import DatabaseService
from metadata.generated.schema.entity.data.database import Database

# Load config.json using the first code cell before publishing.
# Create the metadata destination if it does not already exist.
connection = OpenMetadataConnection.model_validate({
    "hostPort": config.get("OPENMETADATA_URL", "http://localhost:8585/api"),
    "authProvider": "openmetadata",
    "securityConfig": {"jwtToken": config["OPENMETADATA_JWT_TOKEN"]},
})
client = OpenMetadata(connection)
publisher = OpenMetadataPublisher(client)
published_schemas = []
published_tables = {}
published_views = {}
published_profiles = []
published_lineage = []

try:
    service_name, database_name = database_fqn.split(".", 1)
    if client.get_by_name(entity=DatabaseService, fqn=service_name) is None:
        client.create_or_update(CreateDatabaseServiceRequest(
            name=service_name, serviceType="Oracle",
        ))
    if client.get_by_name(entity=Database, fqn=database_fqn) is None:
        client.create_or_update(CreateDatabaseRequest(
            name=database_name, service=service_name,
        ))

    for request in mappings["schemas"]:
        published_schemas.append(publisher.publish_schema(request))

    aliases_by_table = defaultdict(list)
    for synonym in synonyms:
        if not synonym.db_link:
            aliases_by_table[(synonym.table_owner, synonym.table_name)].append(
                mapper.map_synonym(synonym)
            )

    # Create every table before sending foreign keys referencing other tables.
    for request in mappings["tables"]:
        local_constraints = [
            constraint for constraint in (request.tableConstraints or [])
            if constraint.constraintType.value != "FOREIGN_KEY"
        ]
        publisher.publish_table(request.model_copy(update={
            "tableConstraints": local_constraints or None,
        }))

    for source, request in zip(tables, mappings["tables"], strict=True):
        key = (source.schema_name, source.name)
        aliases = aliases_by_table.get(key)
        if aliases:
            request = request.model_copy(update={"aliases": aliases})
        published = publisher.publish_table(request)
        published_tables[key] = published
        entity_ids[key] = published.id.root

    for source, request in zip(views, mappings["views"], strict=True):
        key = (source.schema_name, source.name)
        aliases = aliases_by_table.get(key)
        if aliases:
            request = request.model_copy(update={"aliases": aliases})
        published = publisher.publish_view(request)
        published_views[key] = published
        entity_ids[key] = published.id.root

    column_profiles_by_table: dict[tuple[str, str], list[ColumnProfile]] = defaultdict(list)
    for source, profile in zip(column_statistics, mappings["column_profiles"], strict=True):
        column_profiles_by_table[(source.schema_name, source.table_name)].append(profile)

    whole_table_statistics = [stats for stats in table_statistics if stats.object_type == "TABLE"]
    for source, profile in zip(whole_table_statistics, mappings["table_profiles"], strict=True):
        key = (source.schema_name, source.table_name)
        published_profiles.append(publisher.publish_table_statistics(
            published_tables[key], profile, column_profiles_by_table.get(key)
        ))

    # Rebuild lineage with the server IDs obtained from publishing tables/views.
    pending_lineage = []
    mappings["lineage"] = []
    for dependency in view_dependencies:
        referenced_schema = dependency.referenced_schema_name
        if referenced_schema is None:
            pending_lineage.append(dependency)
            continue
        source_key = (referenced_schema, dependency.referenced_object_name)
        view_key = (dependency.schema_name, dependency.view_name)
        if (
            dependency.referenced_object_type not in {"TABLE", "VIEW", "MATERIALIZED VIEW"}
            or source_key not in entity_ids or view_key not in entity_ids
        ):
            pending_lineage.append(dependency)
            continue
        request = mapper.map_view_dependency(
            dependency, entity_ids[source_key], entity_ids[view_key]
        )
        mappings["lineage"].append(request)
        published_lineage.append(publisher.publish_lineage(request))
finally:
    client.close()

{
    "schemas": len(published_schemas),
    "tables": len(published_tables),
    "views": len(published_views),
    "profiles": len(published_profiles),
    "lineage": len(published_lineage),
    "pending_lineage": len(pending_lineage),
}
